# Flexible paper figures tool

Per-figure block selection, params overrides, inline build, and a dated export —
the flexible counterpart to `analysis_figure_suite.ipynb` (which stays as the
all-at-once driver).

**Workflow**
1. (Optional) Browse & stage blocks → write a paper registry YAML (section 0.5)
2. Point at a registry + params YAML and a run tag
3. Build (or load-cached) event tables once — set `USE_FINALIZED_SACCADES=True` to prefer
   per-block `analysis/saccades/` CSVs from the preprocessing GUI (see also
   `compile_block_saccades.ipynb`)
4. (Optional) Tag bad events in the Saccade Viewer / ROI picker; set `EXCLUDE_VERIFICATION_BAD=True` so Builds drop them
5. For each figure: tick blocks → filter saccades (incl. exclude verification-bad) → tweak params → **Build**
6. Finalize into `outputs/paper_figures_<tag>_<date>_<HH>_<MM>/`

**Mouse / pogona supplementary:** re-run this notebook against
`configs/mouse_M_002_blocks.yaml` + `configs/analysis_params_mouse.yaml`.
Blocks without behavior-state files correctly show as ineligible for Fig 3c/3e/3f.

Requires `ipywidgets`, `PYTHONPATH` including `src` (set below), and the
`eye_repo_mac` (or equivalent) env.


## 0. Setup


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import FileLink, Markdown, display

%matplotlib inline
# Widget Build buttons display figures explicitly; keep pyplot non-interactive
# so matplotlib_inline's post_execute flush cannot paint a second copy.
plt.ioff()

REPO = Path.cwd()
if (REPO / "src" / "eye_tracking_system_tools").is_dir():
    pass
elif (REPO.parent / "src" / "eye_tracking_system_tools").is_dir():
    REPO = REPO.parent
else:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)
CWD = Path.cwd().resolve()

from eye_tracking_system_tools.analysis.block_registry import (
    load_registry,
    registry_summary,
    write_paper_registry,
)
from eye_tracking_system_tools.analysis.event_cache import build_or_load_event_tables
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.figure_catalog import CATALOG
from eye_tracking_system_tools.analysis.jitter_gui import JitterBlockBrowser
from eye_tracking_system_tools.analysis.paper_export import (
    describe_export,
    finalize_paper_export,
    rebuild_from_export,
)
from eye_tracking_system_tools.analysis.paper_gui import (
    PaperContext,
    PaperFigureSelector,
    block_qc_table,
    make_fig1e_selector,
    make_vignette_selector,
)
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir

PAPER_REGISTRY = REPO / "configs" / "paper_blocks_custom.yaml"

def link_path(p: Path):
    p = Path(p).resolve()
    try:
        rel = os.path.relpath(p, CWD)
        display(FileLink(rel))
    except Exception:
        display(Markdown(f"[{p.name}]({p.as_uri()})"))

print("REPO:", REPO)
print("Catalog figures:", ", ".join(CATALOG))


REPO: /Users/nimi/Projects/PETS
Catalog figures: 2c_2d, 2e, 2f, 2g, 2h, 2i, 2j, 3d, 3e, 3f, 3a, 3b, 3c, 1e


## 0.5 Browse & build a paper registry

Use the same filesystem browser as the jitter pipeline to multi-select block
folders, then **Save registry**. With `registry_format="paper"` this writes an
`animals: {name: [paths]}` YAML (what section 1 / `load_registry` expects) —
not the jitter `blocks:` / `mount_type` schema.

Mount tags in the UI are ignored for paper registries; leave them at the default.
Prefer **Add selected** on `block_*` folders (the Scan button looks for jitter
reports, which paper figures do not need).

Skip this section if you already have a registry (e.g. `paper_blocks.yaml`).


In [2]:
# Destination for the paper-format registry written by Save.
# Change the name if you want a dated / tagged file instead of overwriting.
PAPER_REGISTRY = REPO / "configs" / "paper_blocks_custom.yaml"

browser = JitterBlockBrowser(
    PAPER_REGISTRY,
    repo=REPO,
    registry_format="paper",  # animals: {…} — not the jitter blocks: schema
    load_existing=True,       # re-open previous custom registry if present
)
browser


In [3]:
# After clicking Save registry above, confirm what was written.
from collections import defaultdict

print(f"staged: {len(browser.specs)} block(s)")
if browser.specs:
    # Re-save in case the user staged more after the last Save click.
    write_paper_registry(PAPER_REGISTRY, browser.specs)
    print(f"wrote: {PAPER_REGISTRY}")
    link_path(PAPER_REGISTRY)
    by: dict[str, list[str]] = defaultdict(list)
    for s in browser.specs:
        by[s.animal].append(s.block_path.name)
    for animal, names in sorted(by.items()):
        print(f"  {animal}: {', '.join(names)}")
else:
    print("Nothing staged — section 1 can still point at an existing registry.")


staged: 4 block(s)
wrote: /Users/nimi/Projects/PETS/configs/paper_blocks_custom.yaml


/Users/nimi/Projects/PETS/configs/paper_blocks_custom.yaml

  M_002: block_012, block_013, block_014, block_015


## 1. Registry, params, run tag

Edit the paths below. If you built a registry in section 0.5, keep
`REGISTRY = PAPER_REGISTRY`; otherwise point at `paper_blocks.yaml`,
`paper_blocks_dryrun_PV_143.yaml`, `sample_blocks.yaml`, etc.

Empty `TAG` writes to `paper_latest` (overwritable); set a tag to keep a
snapshot under `outputs/paper_<tag>/` for scratch builds. The dated finalize
folder is separate (section 4).


In [4]:
# --- edit me ---
# Prefer the custom registry from section 0.5 when it exists; else the paper cohort.
REGISTRY = (
    PAPER_REGISTRY
    if PAPER_REGISTRY.is_file()
    else REPO / "configs" / "paper_blocks.yaml"
)
# REGISTRY = REPO / "configs" / "paper_blocks_dryrun_PV_143.yaml"
# REGISTRY = REPO / "configs" / "sample_blocks.yaml"
# REGISTRY = REPO / "configs" / "mouse_M_002_blocks.yaml"
PARAMS = REPO / "configs" / "analysis_params.yaml"
# PARAMS = REPO / "configs" / "analysis_params_mouse.yaml"
TAG = ""  # empty → paper_latest
FORCE_REBUILD_EVENTS = False  # True to ignore the event-table cache
KEEP_TRACES = False  # cache stores events only; Fig 2c/2d, 2f, 3* reload traces on Build
# Prefer preprocessing-GUI finalized CSVs under each block's analysis/saccades/
# (falls back to on-the-fly detection when missing). See compile_block_saccades.ipynb.
USE_FINALIZED_SACCADES = True
# ---------------

print("REGISTRY:", REGISTRY)

params = load_params_yaml(PARAMS)
specs = load_registry(REGISTRY)
print(registry_summary(specs))

run = resolve_run_dir(REPO / "outputs", TAG, prefix="paper", default_name="paper_latest")
print("run_dir:", run.run_dir)
print("figures:", run.figures_dir)
print("metadata:", run.metadata_dir)


REGISTRY: /Users/nimi/Projects/PETS/configs/paper_blocks_custom.yaml
{'n_blocks': 4, 'animals': ['M_002'], 'blocks': [{'animal': 'M_002', 'block_num': '012', 'block_path': '/Volumes/Data/Nimrod/experiments/M_002/2026_07_28/block_012'}, {'animal': 'M_002', 'block_num': '013', 'block_path': '/Volumes/Data/Nimrod/experiments/M_002/2026_07_28/block_013'}, {'animal': 'M_002', 'block_num': '014', 'block_path': '/Volumes/Data/Nimrod/experiments/M_002/2026_07_28/block_014'}, {'animal': 'M_002', 'block_num': '015', 'block_path': '/Volumes/Data/Nimrod/experiments/M_002/2026_07_28/block_015'}]}
run_dir: /Users/nimi/Projects/PETS/outputs/paper_latest
figures: /Users/nimi/Projects/PETS/outputs/paper_latest/figures
metadata: /Users/nimi/Projects/PETS/outputs/paper_latest/metadata


## 2. Build event tables (cached) + QC

Detection is the expensive step. Results are cached under
`metadata/event_cache/<sha1>.pkl` keyed on block paths + saccade/binocular params.
A restarted kernel reloads instantly; Fig 2c/2d, 2f, and 3* reload traces for
selected blocks only (kernel 2c/2d needs `k_phi`/`k_theta`/`ms_axis`).


In [5]:
tables, cache_path, from_cache = build_or_load_event_tables(
    specs,
    params,
    run.metadata_dir,
    keep_traces=KEEP_TRACES,
    force=FORCE_REBUILD_EVENTS,
    prefer_finalized=USE_FINALIZED_SACCADES,
)
print(("loaded from cache" if from_cache else "built fresh"), "→", cache_path)
print(
    f"blocks={len(tables.blocks)}  all_saccades={len(tables.all_saccades)}  "
    f"synced_rows={len(tables.synced)}  non_synced={len(tables.non_synced)}"
)

ctx = PaperContext(
    tables,
    run.run_dir,
    registry_path=REGISTRY,
    params_path=PARAMS,
)
qc = block_qc_table(tables)
display(qc)
print(
    "behavior_state:", int(qc["has_behavior_state"].sum()), "/", len(qc),
    " | pix_size:", int(qc["has_pix_size"].sum()), "/", len(qc),
)


[M_002_block_012] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_012] finalized saccades n=2600 synced_pairs≈909 non_synced=782 (from pickle)
[M_002_block_013] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_013] finalized saccades n=4067 synced_pairs≈1240 non_synced=1587 (from pickle)
[M_002_block_014] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_014] finalized saccades n=8853 synced_pairs≈2426 non_synced=4001 (from pickle)
[M_002_block_015] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_015] finalized saccades n=2860 synced_pairs≈760 non_synced=1340 (from pickle)
built fresh → /Users/nimi/Projects/PETS/outputs/paper_latest/metadata/event_cache/8951f130a6d94e4d0522d9061fd44db0291bcb78.pkl
blocks=4  all_saccades=18380  synced_rows=10670  non_synced=7710


,block_key,animal,block,n_saccades_L,n_saccades_R,n_synced_pairs,has_traces,has_behavior_state,has_pix_size,block_path
0,M_002_block_012,M_002,block_012,1289,1311,909,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...
1,M_002_block_013,M_002,block_013,2026,2041,1240,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...
2,M_002_block_014,M_002,block_014,4393,4460,2426,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...
3,M_002_block_015,M_002,block_015,1216,1644,760,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...


behavior_state: 0 / 4  | pix_size: 4 / 4


## 2.5 Saccade verification viewer (optional)

Launch the **Saccade Viewer** on a subsample of pooled events to tag them good/bad.
Tags autosave per block under `analysis/saccade_verification/tags.csv`.

After tagging, enable **Exclude verification-bad** in section 2.7 (or on each figure's
saccade filter) so Builds drop `verification_status=bad` events (keeps `good` and `unset`).

Run after section 2 (`ctx` must exist). Uses the same registry as section 1.
Click **Help** in the window for keyboard shortcuts (no persistent help panel).
Closing the window returns control to the notebook without restarting the kernel.


In [ ]:
from eye_tracking_system_tools.analysis.saccade_viewer.launch import launch_saccade_viewer

# ~3 events per block from the pooled table (adjust head() or use a query).
sample = ctx.tables.all_saccades.groupby("block", sort=False).head(3).copy()
print(f"Launching viewer with {len(sample)} events across {sample['block'].nunique()} blocks")

launch_saccade_viewer(sample, registry_path=REGISTRY)

## 2.6 Threshold-based verification subset

Filter pooled events by scalar columns from `ctx.tables.all_saccades` (amplitude,
duration, head flags, etc.), preview the subset, then launch the Saccade Viewer.

Run after section 2 (`ctx` must exist). Tags still autosave per block under
`analysis/saccade_verification/tags.csv`.

In [ ]:
from eye_tracking_system_tools.analysis.saccade_viewer.selectors import ThresholdSelectorPanel

threshold_panel = ThresholdSelectorPanel(ctx.tables, registry_path=REGISTRY)
threshold_panel

## 2.7 Exclude verification-bad from figures

Events tagged **bad** in the Saccade Viewer (or via the Fig 2f ROI → viewer path) live in
per-block `analysis/saccade_verification/tags.csv`.

Set the flag below **before** creating figure selectors in section 3. Each selector's
**Exclude verification-bad** checkbox seeds from this default; you can still override
per figure. Semantics: drop `bad`, keep `good` and `unset`.


In [ ]:
# Drop viewer-tagged bad events from all subsequent figure Builds (default for selectors).
EXCLUDE_VERIFICATION_BAD = True  # set False to include everything (including tagged bad)

ctx.exclude_verification_bad = bool(EXCLUDE_VERIFICATION_BAD)
print(
    f"ctx.exclude_verification_bad={ctx.exclude_verification_bad} "
    "(re-run figure selector cells after changing this)"
)


## 3. Per-figure selectors

Each cell is independent: tick blocks, optionally filter saccades, edit the YAML
params box, click **Build**.

**Saccade filter** (on every event-based figure):
- **Events** — All / Concurrent (synced pairs) / Monocular
- **Head** — Any / Without head / With head / Labeled only
- **Advanced** — pandas `query` string and per-column truth filters for other
  boolean-like flags

The live count under the filter shows how many events remain after the current
block selection + filter. Filters are stored in the export bundle and replayed
by `rebuild_from_export`.


### Fig 2c / 2d — amp-binned position & velocity


In [6]:
sel_2c_2d = PaperFigureSelector('2c_2d', ctx)
sel_2c_2d


### Fig 2e — amplitude–velocity linear fit


In [ ]:
sel_2e = PaperFigureSelector('2e', ctx)
sel_2e


### Fig 2f — inter-ocular peak-speed coupling (needs traces)

Use the **2f mode** toggle on the selector (also mirrored as `sample_mode` in Params):

- **Legacy (±window)** — paper 2f: max contralateral `angular_speed_r` in ±`contra_sample_ms` around onset
- **Strict (event span)** — max contralateral `angular_speed_r` over `[saccade_on_ms, saccade_off_ms]`

Configure blocks / filters / params → **Build**. Then run the ROI picker cell below for verification on the same setup.


In [ ]:
sel_2f = PaperFigureSelector('2f', ctx)
sel_2f


### Fig 2f — ROI picker (verification)

**Run after Build above.** Reuses the same blocks, saccade filter, params, and export pickle as `sel_2f.result`, including the Legacy/Strict mode from the toggle. Accumulate rectangular ROIs, then launch the viewer on the pooled events.

Drag **corner or edge handles** on the yellow ROI to resize independently.


In [ ]:
from eye_tracking_system_tools.analysis.saccade_viewer.selectors import (
    launch_figure_2f_roi_picker_from_selector,
)

# sample_mode defaults to whatever was Built (Legacy/Strict toggle).
roi_picker = launch_figure_2f_roi_picker_from_selector(
    sel_2f,
    sel_2f.result,
    registry_path=REGISTRY,
)
# After close: roi_picker.selected_events holds the accumulated pool.


### Fig 2f controls — self-eye identity & Δφ split

Diagnostic companions to the selector above. Reuse **`sel_2f`** block ticks, saccade filter, params YAML, and the **Legacy/Strict** toggle (no need to click Build).

1. **Self-eye identity** — run the 2f peak-speed pairing with each eye’s own trace as the “contralateral” sample. Event-mode (mono / binocular) still uses the *real* other eye — otherwise every event would match itself and monocular mode would drop everything. Expect a near-diagonal when the stored `speed_profile_angular` agrees with the reloaded-trace sample (Legacy: ±`contra_sample_ms`; Strict: event `[on, off]`).
2. **Δφ sign split** — real inter-ocular 2f, split by the triggering event’s `delta_phi` (`> 0` vs `< 0`), using the same sample mode.


In [ ]:
import matplotlib.colors as mcolors
from eye_tracking_system_tools.analysis.event_cache import ensure_traces_for_blocks
from eye_tracking_system_tools.analysis.figure_display import show_and_close
from eye_tracking_system_tools.analysis.figures_2f_2h_2i import (
    _contra_has_event,
    _estimate_frame_period_ms,
    _iqr_bounds,
    _resolve_contra_peak,
)
from eye_tracking_system_tools.analysis.pipeline import (
    apply_saccade_filter,
    filter_event_tables,
    with_params,
)


def _tables_for_2f_controls():
    """Same block / filter / params path as a Fig 2f Build from ``sel_2f``."""
    block_keys = sel_2f.selected
    if not block_keys:
        raise RuntimeError("Tick at least one eligible block on the Fig 2f selector above.")
    tables = ensure_traces_for_blocks(ctx.tables, block_keys)
    ctx.tables = tables
    tables = filter_event_tables(tables, block_keys=block_keys)
    tables = apply_saccade_filter(tables, sel_2f.current_saccade_filter())
    overrides = sel_2f._parse_overrides()
    if overrides and "figure_2f" not in overrides:
        overrides = {"figure_2f": overrides}
    tables = with_params(tables, overrides) if overrides else tables
    cfg = dict(tables.params.get("figure_2f", {}))
    # Toggle wins even if params YAML was edited out of sync.
    mode = sel_2f.current_sample_mode()
    if mode:
        cfg["sample_mode"] = mode
    filt = sel_2f.current_saccade_filter()
    print(
        f"[2f controls] blocks={len(block_keys)} events={len(tables.all_saccades)} "
        f"sample_mode={cfg.get('sample_mode', 'contra_window')} "
        f"filter=[{filt.describe() if filt else 'none'}]"
    )
    return tables, cfg


def _collect_2f_speed_pairs(
    tables,
    cfg,
    *,
    contra_source="real",
    eyes=("L", "R"),
    phi_sign=None,
    sample_mode=None,
):
    """
    Collect (x, y) peak-speed pairs with the Fig 2f algorithm.

    contra_source:
        ``"real"`` — other eye (paper 2f; x=right, y=left)
        ``"self"`` — same eye as the event (x=profile peak, y=same-eye sample)
    phi_sign:
        ``None`` keep all; ``"pos"`` / ``"neg"`` filter on event ``delta_phi``.
    sample_mode:
        ``None`` — use ``cfg["sample_mode"]`` (from the Fig 2f Legacy/Strict toggle).
        ``"contra_window"`` — ±``contra_sample_ms`` around onset (legacy).
        ``"event_span"`` — [``saccade_on_ms``, ``saccade_off_ms``] (strict).
    """
    event_mode = str(cfg.get("event_mode", "monocular")).lower()
    if sample_mode is None:
        sample_mode = str(cfg.get("sample_mode", "contra_window")).lower()
    else:
        sample_mode = str(sample_mode).lower()
    contra_win = float(cfg.get("contra_event_window_ms", 100.0))
    contra_sample = float(cfg.get("contra_sample_ms", 51.0))
    exclude = {str(a) for a in cfg.get("exclude_animals", [])}
    eyes = {str(e).upper() for e in eyes}

    df = tables.all_saccades.copy()
    if exclude:
        df = df[~df["animal"].astype(str).isin(exclude)]
    if cfg.get("require_head_stationary"):
        if "head_movement" not in df.columns or df["head_movement"].notna().sum() == 0:
            print("[2f controls] WARNING: require_head_stationary but no labels; keeping all rows")
        else:
            before = len(df)
            df = df[df["head_movement"] == False]  # noqa: E712
            print(f"[2f controls] head_stationary filter: {before} → {len(df)}")
    if phi_sign is not None:
        if "delta_phi" not in df.columns:
            raise ValueError("delta_phi missing — cannot split by phi sign")
        phi = pd.to_numeric(df["delta_phi"], errors="coerce")
        before = len(df)
        if phi_sign == "pos":
            df = df.loc[phi > 0]
        elif phi_sign == "neg":
            df = df.loc[phi < 0]
        else:
            raise ValueError(f"phi_sign must be None/'pos'/'neg', got {phi_sign!r}")
        print(f"[2f controls] delta_phi {phi_sign}: {before} → {len(df)}")

    block_map = tables.block_dict
    onset_cache = {}
    for key, bundle in block_map.items():
        for eye, ev in (("L", bundle.l_saccades), ("R", bundle.r_saccades)):
            if ev is None or ev.empty or "saccade_on_ms" not in ev.columns:
                onset_cache[(key, eye)] = np.array([], dtype=float)
            else:
                onset_cache[(key, eye)] = ev["saccade_on_ms"].to_numpy(dtype=float)

    xs, ys, animals = [], [], []
    n_skip_mode = n_skip_profile = n_skip_eye = 0
    for _, row in df.iterrows():
        animal = str(row["animal"])
        block_key = f"{animal}_block_{row['block']}"
        bundle = block_map.get(block_key)
        if bundle is None:
            continue
        eye = str(row["eye"]).upper()
        if eye not in eyes:
            n_skip_eye += 1
            continue
        t0 = float(row["saccade_on_ms"])
        other = "R" if eye == "L" else "L"
        is_binocular = _contra_has_event(onset_cache[(block_key, other)], t0, contra_win)
        if event_mode == "monocular" and is_binocular:
            n_skip_mode += 1
            continue
        if event_mode == "binocular" and not is_binocular:
            n_skip_mode += 1
            continue

        if eye == "L":
            ipsi_df = bundle.left
            real_contra_df = bundle.right
        else:
            ipsi_df = bundle.right
            real_contra_df = bundle.left
        contra_df = ipsi_df if contra_source == "self" else real_contra_df

        frame_ms = _estimate_frame_period_ms(ipsi_df, t0)
        sp = row.get("speed_profile_angular", None)
        if sp is None or len(sp) == 0 or not np.isfinite(np.nanmax(sp)):
            n_skip_profile += 1
            continue
        ipsi_peak = float(np.nanmax(sp)) / frame_ms
        contra_peak = _resolve_contra_peak(
            contra_df,
            row,
            sample_mode=sample_mode,
            contra_sample_ms=contra_sample,
            frame_ms=frame_ms,
        )
        if not np.isfinite(contra_peak):
            n_skip_profile += 1
            continue

        if contra_source == "self":
            xs.append(ipsi_peak)
            ys.append(contra_peak)
        elif eye == "L":
            xs.append(contra_peak)
            ys.append(ipsi_peak)
        else:
            xs.append(ipsi_peak)
            ys.append(contra_peak)
        animals.append(animal)

    x = np.asarray(xs, dtype=float)
    y = np.asarray(ys, dtype=float)
    animals_arr = np.asarray(animals)
    print(
        f"[2f controls] contra={contra_source} eyes={sorted(eyes)} phi={phi_sign} "
        f"sample={sample_mode} event_mode={event_mode} kept={x.size} skip_mode={n_skip_mode} "
        f"skip_profile={n_skip_profile} skip_eye={n_skip_eye}"
    )
    if x.size == 0:
        return x, y, np.asarray([], dtype=float)

    u, c = np.unique(animals_arr, return_counts=True)
    wmap = {a: (len(animals_arr) / (len(u) * cnt)) for a, cnt in zip(u, c)}
    weights = np.array([wmap[a] for a in animals_arr], dtype=float)

    iqr_mult = float(cfg.get("iqr_multiplier", 60.0))
    x_lo, x_hi = _iqr_bounds(x, iqr_mult)
    y_lo, y_hi = _iqr_bounds(y, iqr_mult)
    keep = (x >= x_lo) & (x <= x_hi) & (y >= y_lo) & (y <= y_hi)
    return x[keep], y[keep], weights[keep]


def _plot_2f_style_hist(x, y, weights, cfg, *, xlabel, ylabel, suptitle, stem):
    if x.size == 0:
        print(f"[2f controls] no points for {stem} — skip plot")
        return None
    if x.size >= 3:
        r = float(np.corrcoef(x, y)[0, 1])
        mad = float(np.median(np.abs(y - x)))
        print(f"[2f controls] {stem}: n={x.size}  pearson_r={r:.4f}  median_|y-x|={mad:.4g}")
    else:
        print(f"[2f controls] {stem}: n={x.size}")

    bins = int(cfg.get("bins", 60))
    macro_range = tuple(cfg.get("macro_range", [0.0, 0.5]))
    micro_range = tuple(cfg.get("micro_range", [0.0, 0.1]))
    macro_ticks = cfg.get("macro_tick_list", [0, 0.25, 0.5])
    micro_ticks = cfg.get("micro_tick_list", [0, 0.05, 0.1])

    def _hist(rng):
        xbins = np.linspace(rng[0], rng[1], bins)
        ybins = np.linspace(rng[0], rng[1], bins)
        counts, xedges, yedges = np.histogram2d(x, y, bins=[xbins, ybins], weights=weights)
        norm = counts / counts.sum() if counts.sum() > 0 else counts
        return xedges, yedges, norm

    turbo = plt.get_cmap("turbo", 256)
    colors = turbo(np.linspace(0, 1, 256))
    colors[0] = np.array([1, 1, 1, 1])
    cmap = mcolors.ListedColormap(colors)

    fig, axs = plt.subplots(1, 2, figsize=(5.2, 2.5), dpi=150, constrained_layout=True)
    for ax, rng, title, ticks in (
        (axs[0], macro_range, "Macro", macro_ticks),
        (axs[1], micro_range, "Micro", micro_ticks),
    ):
        xedges, yedges, norm = _hist(rng)
        ax.pcolormesh(
            xedges,
            yedges,
            norm.T,
            cmap=cmap,
            vmin=0,
            vmax=float(np.nanmax(norm) or 1),
            shading="flat",
        )
        ax.set_xlim(*rng)
        ax.set_ylim(*rng)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)
        ax.plot([rng[0], rng[1]], [rng[0], rng[1]], ls="--", color="gray", lw=1)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_box_aspect(1)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if ax.collections:
            ax.collections[0].set_rasterized(True)
    fig.suptitle(suptitle, fontsize=10)
    out_pdf = run.figures_dir / f"{stem}.pdf"
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
    show_and_close(fig, show=True)
    print("wrote", out_pdf)
    return out_pdf


# --- 1. Self-eye identity (L vs L, R vs R); sample_mode from Fig 2f toggle ---
_ctrl_tables, _ctrl_cfg = _tables_for_2f_controls()
_mode = str(_ctrl_cfg.get("sample_mode", "contra_window"))
_ylabel_suffix = (
    "same-eye ±window sample" if _mode == "contra_window" else "same-eye event-span sample"
)
for _eye, _label in (("L", "Left"), ("R", "Right")):
    _x, _y, _w = _collect_2f_speed_pairs(
        _ctrl_tables,
        _ctrl_cfg,
        contra_source="self",
        eyes=(_eye,),
    )
    _plot_2f_style_hist(
        _x,
        _y,
        _w,
        _ctrl_cfg,
        xlabel=f"{_label} profile peak [deg/ms]",
        ylabel=f"{_label} {_ylabel_suffix} [deg/ms]",
        suptitle=f"Fig 2f control — {_label} eye vs itself ({_mode})",
        stem=f"figure_2f_control_self_{_eye}",
    )


In [ ]:
# --- 2. Real inter-ocular 2f split by Δφ sign (sample_mode from Fig 2f toggle) ---
_ctrl_tables, _ctrl_cfg = _tables_for_2f_controls()
_mode = str(_ctrl_cfg.get("sample_mode", "contra_window"))
for _sign, _title, _stem in (
    ("pos", r"$\Delta\phi > 0$", "figure_2f_control_phi_pos"),
    ("neg", r"$\Delta\phi < 0$", "figure_2f_control_phi_neg"),
):
    _x, _y, _w = _collect_2f_speed_pairs(
        _ctrl_tables,
        _ctrl_cfg,
        contra_source="real",
        phi_sign=_sign,
    )
    _plot_2f_style_hist(
        _x,
        _y,
        _w,
        _ctrl_cfg,
        xlabel="Right max V [deg/ms]",
        ylabel="Left max V [deg/ms]",
        suptitle=f"Fig 2f control — {_title} ({_mode})",
        stem=_stem,
    )


### Fig 2g — amplitude distributions


In [ ]:
sel_2g = PaperFigureSelector('2g', ctx)
sel_2g


### Fig 2h — endpoint heatmaps


In [ ]:
sel_2h = PaperFigureSelector('2h', ctx)
sel_2h


### Fig S3 — peak-speed coupling still vs moving

Same pairing as Fig 2f (all saccades, ±51 ms contra peak, equal-animal weights), split by `head_movement`. The two panels share a colorbar; it is **not** shared with Fig 2f.


In [ ]:
sel_s3 = PaperFigureSelector('s3', ctx)
sel_s3


### Fig 2i — polar direction histograms


In [ ]:
sel_2i = PaperFigureSelector('2i', ctx)
sel_2i


### Fig 2j — orientation tuning


In [ ]:
sel_2j = PaperFigureSelector('2j', ctx)
sel_2j


### Fig 3d — inter-saccade interval densities


In [ ]:
sel_3d = PaperFigureSelector('3d', ctx)
sel_3d


### Fig 3e — pupil by behavioral state


In [ ]:
sel_3e = PaperFigureSelector('3e', ctx)
sel_3e


### Fig 3f — z-scored pupil state difference


In [ ]:
sel_3f = PaperFigureSelector('3f', ctx)
sel_3f


### Fig 2b — simultaneous φ/θ traces (N/T/D/V on example trajectories)

Time-trace panel. Anatomical nasal/temporal/dorsal/ventral labels are applied when replotting archived saccade-example trajectories (`2b_examples`).


In [ ]:
sel_2b = make_vignette_selector("2b", ctx, start_s=0.0, end_s=8.0)
sel_2b


In [ ]:
sel_2b_examples = PaperFigureSelector("2b_examples", ctx)
sel_2b_examples


### Fig 3a — quiet vignette (single block + time window)


In [ ]:
sel_3a = make_vignette_selector("3a", ctx, start_s=210.0, end_s=240.0)
sel_3a


### Fig 3b — active vignette


In [ ]:
sel_3b = make_vignette_selector("3b", ctx, start_s=310.0, end_s=340.0)
sel_3b


### Fig 3c — full vignette with state / rates


In [ ]:
sel_3c = make_vignette_selector("3c", ctx, start_s=200.0, end_s=415.0)
sel_3c


### Fig 1e — camera jitter (from a finalized jitter export)

Point at a folder produced by `jitter_mount_pipeline.ipynb` section 6
(`jitter_comparison_figures_<tag>_<date>_…`), or its `jitter_comparison_data.pickle`.


In [ ]:
# Optional default: newest jitter export under outputs/, else leave blank
_jitter_candidates = sorted((REPO / "outputs").glob("jitter_comparison_figures_*"), reverse=True)
JITTER_BUNDLE = _jitter_candidates[0] if _jitter_candidates else None
print("default jitter bundle:", JITTER_BUNDLE)

sel_1e = make_fig1e_selector(ctx, JITTER_BUNDLE)
sel_1e


## 4. Finalize export

Creates `outputs/paper_figures_<tag>_<YYYYmmdd>_<HH>_<MM>/` with the PDFs that
were built, plus `figure_specs.pickle`, `selections.csv`, and `manifest.yaml`.
Only selections + params + registry pointers are stored — rebuild re-runs from
source blocks.


In [ ]:
EXPORT_TAG = TAG or "dryrun"  # edit
EXPORT_NOTES = ""

if not ctx.builds:
    raise RuntimeError("Build at least one figure in section 3 before finalizing.")

result = finalize_paper_export(
    ctx.builds,
    REPO / "outputs",
    tag=EXPORT_TAG,
    registry_path=REGISTRY,
    params_path=PARAMS,
    notes=EXPORT_NOTES,
    scratch_figures_dir=run.figures_dir,
)
print(result)
link_path(result.export_dir)
link_path(result.manifest_path)
display(describe_export(result.export_dir))


## 5. Rebuild from export (demo)

Re-runs each recorded figure from its stored selection and params. Needs the
same data volumes mounted and event tables covering those blocks.


In [ ]:
# Point at the export you just made (or any earlier one)
EXPORT_PATH = result.export_dir  # or Path(".../paper_figures_...")

rebuild_dir = run.run_dir / "rebuild_demo"
rebuild_dir.mkdir(parents=True, exist_ok=True)
rebuilt = rebuild_from_export(EXPORT_PATH, tables, rebuild_dir, show=False)
print(f"rebuilt {len(rebuilt)} figure(s) → {rebuild_dir}")
for fig_id, paths in rebuilt.items():
    print(fig_id, ":", ", ".join(Path(p).name for p in paths.values() if str(p).endswith('.pdf')))


## Notes

| Topic | Detail |
|---|---|
| Suite notebook | `analysis_figure_suite.ipynb` remains the all-at-once driver |
| Scratch vs finalize | Scratch: `outputs/paper_<tag|latest>/`; finalize: dated `paper_figures_*` |
| Event cache | `metadata/event_cache/<sha1>.pkl` — events only, no traces |
| Fig 1e | Replots a finalized jitter export (µm); single source of truth with the jitter tool |
| Mouse registry | `configs/mouse_M_002_blocks.yaml` + `analysis_params_mouse.yaml` |
| FileLink cwd | Kernel cwd is `development/`; helpers above use `os.path.relpath` |
| Verification-bad | Per-block `tags.csv`; enable via §2.7 / selector checkbox (`exclude_bad`) |
